# Key input info

In [1]:
root_name = '251026'
root_dir = rf"../../data/output/mosa/{root_name}"

In [2]:
import os
os.environ['GDAL_DATA'] = r'/usr/share/gdal'
from test4plot2 import plot_double, clearing, plot_cross_tab
from analysis import basic_stat, find_knee
from spatial_plot import plot_compared_cs, plot_zone_cs, match_cs, plot_delta_zone
from delay_analysis import plot_demand_heatmap
import pandas as pd
import geopandas as gpd
from pyproj import CRS
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from IPython.display import IFrame
warnings.simplefilter("ignore")

In [4]:
pd.options.display.float_format = '{:.1f}'.format  # 所有浮点数保留零位小数
means_stat_columns = ['bulit_num', 'fast_num_avg', 'slow_num_avg','large_num','medium_num','small_num','large_model','medium_model','small_model']
MEANS_PARETO = []
KNEES = []
SUPPLE = []
for name in os.listdir(root_dir):
    full_path = os.path.join(root_dir, name)
    if os.path.isdir(full_path) and name.endswith(f'_{root_name}'):
        city_name = name[:-len(f'_{root_name}')]
        print(city_name)
        cs_gdf = gpd.read_file(rf'../../data/input/cs_gdf/{city_name}.shp', crs=CRS.from_epsg(4507))  # 充电站
        cs_gdf = cs_gdf.drop_duplicates(subset=['node_id'], keep='first').reset_index(drop=True)
        cs_num = len(cs_gdf)
        OBJ_G = pd.read_csv(rf"../../data/output/mosa/{root_name}/{city_name}_{root_name}/inf_archive_objs.csv")
        VAR_G = pd.read_csv(rf"../../data/output/mosa/{root_name}/{city_name}_{root_name}/inf_archive_vars.csv")
        CV_G = pd.read_csv(rf"../../data/output/mosa/{root_name}/{city_name}_{root_name}/inf_archive_cvs.csv")
        OBJ_G = clearing(OBJ_G)
        print(f'{len(OBJ_G)}, {len(VAR_G)}')
        STAT_G = basic_stat(OBJ_G, VAR_G, CV_G, cs_num)
        MEANS_G = STAT_G[['bulit_num', 'fast_num_avg', 'slow_num_avg','large_num','medium_num','small_num']].mean()
        MODES_G = STAT_G.iloc[:, -3:].mode().iloc[0]
        OVER_G = pd.DataFrame([MEANS_G.tolist() + MODES_G.tolist()], 
                      columns=means_stat_columns)
        # 添加city_name和cs_num列并调整列顺序
        OVER_G['city_name'] = city_name
        OVER_G['candidate_num'] = cs_num
        OVER_G = OVER_G[['city_name', 'candidate_num'] + means_stat_columns]
        MEANS_PARETO.append(OVER_G)

        knee_no_G = find_knee(STAT_G)
        row_G = STAT_G.loc[STAT_G['solution_no'] == knee_no_G]
        row_G['city_name'] = city_name
        row_G['candidate_num'] = cs_num
        KNEES.append(row_G)

        vs_parking_df = pd.read_csv(rf"../../data/input/vs_parking_nodeid//{city_name}.csv")
        vehicle_count = vs_parking_df['v_name'].nunique()
        operational_distance = vs_parking_df['distance'].sum()
        SUPPLE.append([city_name,vehicle_count,operational_distance])
        
        
MEANS_PARETO = pd.concat(MEANS_PARETO, ignore_index=True).sort_values('candidate_num', ascending=False)
KNEES = pd.concat(KNEES, ignore_index=True).sort_values('candidate_num', ascending=False)
KNEES['built_ratio'] = KNEES['bulit_num'] / KNEES['candidate_num']
front = ['city_name']
rest = [c for c in KNEES.columns if c not in front]
# 重新排序
KNEES = KNEES[ front + rest ]
SUPPLE = pd.DataFrame(SUPPLE, columns=['city_name','vehicle_count','operational_distance'])

七台河市
118, 118
三亚市
210, 211
三明市
191, 194
三门峡市
138, 138
上海市
356, 384
上饶市
185, 185
东莞市
254, 293
东营市
235, 238
中山市
263, 297
临汾市
138, 138
丹东市
162, 162
乌兰察布市
49, 49
乌鲁木齐市
247, 278
九江市
170, 171
云浮市
108, 108
佛山市
302, 346
佳木斯市
150, 150
保定市
219, 219
信阳市
225, 225
六安市
200, 217
六盘水市
197, 197
兰州市
253, 270
内江市
183, 183
包头市
199, 200
北京市
277, 282
北海市
140, 140
十堰市
207, 215
南京市
317, 346
南充市
230, 231
南宁市
229, 237
南平市
201, 203
南昌市
249, 286
南通市
236, 268
南阳市
341, 341
厦门市
253, 292
双鸭山市
110, 110
台州市
190, 192
吉安市
153, 153
吉林市
216, 224
吕梁市
119, 119
周口市
163, 163
呼和浩特市
306, 306
咸宁市
88, 88
咸阳市
276, 276
哈密市
152, 152
哈尔滨市
300, 371
商丘市
271, 271
嘉兴市
225, 231
四平市
106, 106
固原市
14, 14
大同市
243, 243
大庆市
204, 204
天水市
270, 270
天津市
290, 332
太原市
228, 236
威海市
183, 204
娄底市
100, 100
宁德市
174, 174
宁波市
303, 330
安庆市
190, 190
安阳市
78, 78
安顺市
132, 132
宜宾市
248, 248
宜昌市
216, 216
宜春市
156, 156
宝鸡市
239, 239
宿州市
170, 170
宿迁市
223, 223
岳阳市
247, 247
常州市
261, 273
常德市
241, 241
平顶山市
193, 193
广州市
271, 271
庆阳市
47, 47
廊坊市
255, 255
开封市
215, 215
张家口市
227,

## 1. Overvall

### 1.1 Average statistics of Pareto solutions

In [7]:
MEANS_PARETO = MEANS_PARETO.reset_index(drop=True)
MEANS_PARETO

,city_name,candidate_num,bulit_num,fast_num_avg,slow_num_avg,large_num,medium_num,small_num,large_model,medium_model,small_model
0,北京市,1551,803.9,1.6,9.1,371.3,18.6,9.7,4.0,2.0,1.0
1,重庆市,1260,616.2,1.6,8.5,63.4,22.3,0.4,4.0,2.0,1.0
2,上海市,1241,546.0,1.7,10.0,178.8,31.3,0.1,4.0,2.0,1.0
3,杭州市,1098,468.0,0.8,7.0,360.2,79.4,4.0,4.0,2.0,1.0
4,广州市,944,440.6,1.6,9.4,179.1,14.8,3.0,4.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
219,忻州市,14,5.4,4.1,10.7,4.0,0.0,0.0,4.0,2.0,0.0
220,庆阳市,14,6.0,10.1,17.1,0.2,0.0,0.0,4.0,2.0,0.0
221,乌兰察布市,12,5.5,4.1,13.9,0.0,8.3,0.0,4.0,2.0,1.0
222,固原市,11,4.2,0.3,3.9,0.3,0.0,0.0,4.0,2.0,1.0


### 1.2 Average statistics of knee points

In [10]:
KNEES['built_ratio'] = KNEES['built_ratio'].map('{:.2f}'.format)
KNEES = KNEES.reset_index(drop=True)
KNEES
KNEES.to_csv(r'E:/Manufacture/Python/cnbus/data/224cities_output.csv')

### 1.3 Key indicators from knee points

In [15]:
SUPPLE = pd.merge(SUPPLE,KNEES,on='city_name')
SUPPLE['vehicle_count'] = SUPPLE['vehicle_count']+SUPPLE['large_num']+SUPPLE['medium_num']+SUPPLE['small_num']
SUPPLE['cost_per_km'] = SUPPLE['obj1']/SUPPLE['operational_distance']
SUPPLE['emission_per_km'] = SUPPLE['obj2']/SUPPLE['operational_distance']
SUPPLE['station_per_km'] = SUPPLE['bulit_num']/SUPPLE['operational_distance']
SUPPLE['charger_per_km'] = (SUPPLE['fast_num_avg']+SUPPLE['slow_num_avg'])*SUPPLE['bulit_num']/SUPPLE['operational_distance']
SUPPLE['ev_per_km'] = SUPPLE['vehicle_count'] / SUPPLE['operational_distance']
SUPPLE = SUPPLE[['city_name','cost_per_km','emission_per_km','station_per_km','charger_per_km','ev_per_km']]

KeyError: 'vehicle_count'

In [21]:
pd.options.display.float_format = '{:.10f}'.format
SUPPLE

,city_name,cost_per_km,emission_per_km,station_per_km,charger_per_km,ev_per_km,solution_no,obj1,obj2,bulit_num,fast_num_avg,slow_num_avg,large_num,medium_num,small_num,large_model,medium_model,small_model,candidate_num,built_ratio
0,上海市,0.0000020041,0.0000643924,0.0000004296,0.0000020673,0.0000165867,SA145287,2131.7026414970,68493.3436384737,457,0.1225382932,4.6892778993,4,1,0,4,2,1,1241,0.37
1,乌鲁木齐市,0.0000023505,0.0000674770,0.0000002128,0.0000022010,0.0000203992,SA146838,386.5803382430,11097.7617286352,35,0.4857142857,9.8571428571,0,0,0,4,2,1,194,0.18
2,北京市,0.0000022674,0.0000670682,0.0000004410,0.0000022451,0.0000184820,SA146903,3413.5460655350,100972.3606564851,664,0.2078313253,4.8825301205,9,0,2,4,2,1,1551,0.43
3,南京市,0.0000023488,0.0000657308,0.0000004332,0.0000026011,0.0000185719,SA146660,1198.3156951548,33534.4459442144,221,0.3484162896,5.6561085973,2,0,0,4,2,1,643,0.34
4,哈尔滨市,0.0000024312,0.0000662549,0.0000003774,0.0000029034,0.0000195917,SA145401,650.6280509382,17730.7280259584,101,0.3663366337,7.3267326733,2,0,0,4,2,1,341,0.30
5,威海市,0.0000017391,0.0000660853,0.0000003872,0.0000023435,0.0000129710,SA145021,170.6835516949,6485.7409553219,38,0.2894736842,5.7631578947,0,0,0,4,2,1,189,0.20
6,广州市,0.0000022705,0.0000648259,0.0000004109,0.0000021428,0.0000176408,SA146159,2110.6234680394,60262.5123715960,382,0.1413612565,5.0732984293,4,0,1,4,2,1,944,0.40
7,成都市,0.0000021450,0.0000641468,0.0000004703,0.0000024593,0.0000181449,SA145325,1432.1951865144,42829.6162568184,314,0.1369426752,5.0923566879,0,0,0,4,2,1,901,0.35
8,昆明市,0.0000028181,0.0000651505,0.0000004372,0.0000025900,0.0000272285,SA146758,341.6490951469,7898.4123642809,53,0.0000000000,5.9245283019,0,0,0,4,2,1,253,0.21
9,武汉市,0.0000020763,0.0000653248,0.0000004948,0.0000019951,0.0000156819,SA144304,1057.3693410923,33266.6809866699,252,0.0912698413,3.9404761905,2,0,0,4,2,1,701,0.36


## 2. Pareto front

In [13]:
city_name = '北京市'
OBJ_G = pd.read_csv(rf"../../data/output/{root_name}/{city_name}_{root_name}/archive_objs.csv")
OBJ_G = clearing(OBJ_G)
plot_double(OBJ_G,save_pic=False)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/output/250630/北京市_250630/archive_objs.csv'